In [ ]:
# Parameters
H5_PATH = "data.h5"   # 기본값, 필요 시 최신 런 폴더 data.h5로 변경
AGG_METHOD = "sum"    # 프레임 -> 스칼라 변환 방식 ("sum" 또는 "mean")
BASELINE_Q = 0.95     # I_off 추정용 상위 분위수
STAT_ACROSS_SWEEPS = "mean"  # 주파수별 집계 방식 ("mean" 또는 "median")
DROP_WARMUP = True    # 첫 스윕 드롭 여부
MIN_CYCLES_PER_FREQ = 2
SMOOTHING_WIN = None  # 시각화용 smoothing 창 크기 (예: 3, None이면 사용안함)


In [ ]:
import h5py, numpy as np, pandas as pd, matplotlib.pyplot as plt, os

with h5py.File(H5_PATH, "r") as f:
    print("Keys in HDF5:", list(f.keys()))
    roi = f["roi"][:]
    frame_num = f["frame_num"][:]
    freq_hz = f["freq_hz"][:]

print("roi shape:", roi.shape)
print("frame_num shape:", frame_num.shape)
print("freq_hz shape:", freq_hz.shape)


In [ ]:
freq_GHz = freq_hz / 1e9
print("Unique freqs:", len(np.unique(freq_GHz)))
print("Frames:", len(frame_num))
print("ROI pixel count per frame:", roi.shape[1]*roi.shape[2])


In [ ]:
if DROP_WARMUP:
    first_val = freq_GHz[0]
    reset_indices = np.where(freq_GHz == first_val)[0]
    if len(reset_indices) > 1:
        cut = reset_indices[1]
        roi = roi[cut:]
        frame_num = frame_num[cut:]
        freq_GHz = freq_GHz[cut:]
        print(f"Warm-up cycle dropped, start from index {cut}")
    else:
        print("No warm-up cycle detected")


In [ ]:
if AGG_METHOD == "sum":
    I = roi.reshape(len(roi), -1).sum(axis=1)
else:
    I = roi.reshape(len(roi), -1).mean(axis=1)


In [ ]:
I_sorted = np.sort(I)
cut = int((1-BASELINE_Q)*len(I_sorted))
I_off = np.median(I_sorted[-cut:]) if cut>0 else np.max(I_sorted)
print("I_off:", I_off)


In [ ]:
df = pd.DataFrame({"freq_GHz": freq_GHz, "I": I})
if STAT_ACROSS_SWEEPS == "mean":
    grouped = df.groupby("freq_GHz")["I"].mean()
else:
    grouped = df.groupby("freq_GHz")["I"].median()

freqs = grouped.index.values
I_avg = grouped.values


In [ ]:
contrast_pct = (I_off - I_avg)/I_off * 100.0
contrast_pl = I_avg/I_off


In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(6,8), sharex=True)

pl_norm = I_avg / np.max(I_avg)
ax1.plot(freqs, pl_norm, marker='o')
ax1.set_ylabel("PL (norm.)")
ax1.set_ylim(pl_norm.min()-0.01, 1.0)

ax2.plot(freqs, contrast_pl, marker='o')
ax2.set_ylabel("1 - ΔPL/PL (norm.)")
ax2.set_xlabel("Frequency (GHz)")
ax2.set_ylim(contrast_pl.min()-0.01, 1.0)

plt.tight_layout()
plt.savefig("odmr_contrast.png")
plt.show()


In [ ]:
df_out = pd.DataFrame({
    "freq_GHz": freqs,
    "PL_norm": pl_norm,
    "contrast_pct": contrast_pct,
    "contrast_pl": contrast_pl
})
df_out.to_csv("odmr_contrast.csv", index=False)
print("Saved odmr_contrast.csv")
